In [10]:
# import os
# from pymongo import MongoClient
# from pymongo.server_api import ServerApi

# # Connect
# client = MongoClient(os.environ['MONGO_URI'], server_api=ServerApi('1'))
# db = client["campomaq"]

# # Original and backup collection
# src_collection = db["cm_catalog"]
# backup_collection = db["cm_catalog_backup"]

# # Drop existing backup (optional, if you want overwrite behavior)
# backup_collection.drop()

# # Copy documents
# pipeline = [{"$match": {}}, {"$out": "cm_catalog_backup"}]
# src_collection.aggregate(pipeline)

# print("Backup complete: cm_catalog -> cm_catalog_backup")


In [4]:
import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
import pandas as pd
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from bson.decimal128 import Decimal128
# Load environment variables
load_dotenv(find_dotenv())

# SQL and MongoDB connections
engine = create_engine(os.environ['SQLSERVER_URI'])
client = MongoClient(os.environ['MONGO_URI'], server_api=ServerApi('1'))
cm_env = os.environ.get('CM_ENV', 'dev')

# TOP SALES PRODUCTS

In [5]:
# TOP SALES PRODUCTS
cm_env = os.environ.get('CM_ENV', 'dev')
if cm_env == 'prod':
    collection = client["campomaq"]["cm_top_sales_products"]
else:
    collection = client["campomaq_test"]["cm_top_sales_products"]

# Read all documents from the collection
cursor = collection.find({})
df_top_sales_prods = pd.DataFrame(list(cursor))

In [6]:
df_top_sales_prods = df_top_sales_prods[["product_code", "main_boost", "low_value_flag", "popularity"]]
df_top_sales_prods


,product_code,main_boost,low_value_flag,popularity
0,700M-491SI,0,1,0.022477
1,1600-0000,1,0,0.016804
2,1300-0248,1,0,0.010149
3,408-0055,0,0,0.003484
4,435-0000,1,0,0.008157
...,...,...,...,...
2611,1300-0289,0,0,0.000158
2612,700B-113,0,0,0.000075
2613,403-0081,0,0,0.000064
2614,700M-674,0,0,0.000047


# PRODUCT INFO

In [22]:
# PRODUCTS
cm_env = os.environ.get('CM_ENV', 'dev')
if cm_env == 'prod':
    collection = client["campomaq"]["cm_products"]
else:
    collection = client["campomaq_test"]["cm_products"]

# Read all documents from the collection
cursor = collection.find({})
df_products = pd.DataFrame(list(cursor))

df_products = df_products[~df_products["category_name"].str.lower().str.contains("servicios", na=False)]
df_products = df_products[~df_products["product_name"].str.lower().str.contains("varios", na=False)]

In [23]:
df_products = df_products[["product_id","product_code", "price_cash", "price_credit", "price_card"]]
df_products

,product_id,product_code,price_cash,price_credit,price_card
0,0,0,0.0000,120.000,0.0000
1,3368,700M-024,111.1525,114.487,116.7101
2,4167,700M-121,0.0000,0.000,0.0000
3,1,101-0066SI,902.9126,930.000,976.5000
4,803,305-0106SI,776.6990,800.000,840.0000
...,...,...,...,...,...
4692,4691,700B-547,9.0000,9.450,10.0000
4693,4714,700B-112,38.0000,39.000,40.9500
4694,4715,700B-111,33.7500,35.500,37.3000
4695,4756,308-0014SI,16.4000,17.250,18.1100


# IMAGE LINKS

In [26]:
# Select the database and collection
cm_env = os.environ.get('CM_ENV', 'dev')
if cm_env == 'prod':
    collection = client["campomaq"]["cm_images"]
else:
    collection = client["campomaq_test"]["cm_images"]

# Read all documents from the collection
cursor = collection.find({})
df_product_images = pd.DataFrame(list(cursor))

In [27]:
df_product_images = df_product_images[['product_id','link']]
df_product_images

,product_id,link
0,1,[https://campomaq.blob.core.windows.net/campom...
1,10,[https://campomaq.blob.core.windows.net/campom...
2,14,[https://campomaq.blob.core.windows.net/campom...
3,19,[https://campomaq.blob.core.windows.net/campom...
4,20,[https://campomaq.blob.core.windows.net/campom...
...,...,...
2382,4193,[https://campomaq.blob.core.windows.net/campom...
2383,4232,[https://campomaq.blob.core.windows.net/campom...
2384,4233,[https://campomaq.blob.core.windows.net/campom...
2385,4285,[https://campomaq.blob.core.windows.net/campom...


In [21]:
df_products.count()

product_id      9390
product_code    9390
price_cash      9390
price_credit    9390
price_card      9390
dtype: int64

In [28]:
merged_df = pd.merge(df_products, df_top_sales_prods, left_on='product_code', right_on='product_code', how='right')
merged_df = pd.merge(merged_df, df_product_images, left_on='product_id', right_on='product_id', how='left')
# merged_df = merged_df.drop(columns=['product_id'])


In [29]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2616 entries, 0 to 2615
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      2610 non-null   float64
 1   product_code    2616 non-null   object 
 2   price_cash      2610 non-null   float64
 3   price_credit    2610 non-null   float64
 4   price_card      2610 non-null   float64
 5   main_boost      2616 non-null   int64  
 6   low_value_flag  2616 non-null   int64  
 7   popularity      2616 non-null   float64
 8   link            1494 non-null   object 
dtypes: float64(5), int64(2), object(2)
memory usage: 184.1+ KB


In [30]:
# final_df.info()
import numpy as np

def prepare_catalog_df(df: pd.DataFrame) -> pd.DataFrame:
    # --- Handle nulls ---
    # Replace NaN with None (Mongo stores None as null)
    df = df.replace({np.nan: None})
    
    # Ensure link is always a list (empty list if null)
    if "link" in df.columns:
        df["link"] = df["link"].apply(lambda x: x if isinstance(x, list) else ([] if x is None else [x]))
    
    # Replace nulls in numeric fields with 0.0
    numeric_fill_zero = ["main_boost", "low_value_flag", "popularity"]
    for col in numeric_fill_zero:
        if col in df.columns:
            df[col] = df[col].fillna(0.0)
    
    # Ensure booleans are actual Python bool
    bool_fields = ["new_product", "show_in_app","is_spare_part"]
    for field in bool_fields:
        if field in df.columns:
            df[field] = df[field].astype(bool)
    
    # --- Sort by popularity ---
    if "popularity" in df.columns:
        df = df.sort_values(by="popularity", ascending=False).reset_index(drop=True)
    
    return df

final_mongo_df = prepare_catalog_df(merged_df)

/var/folders/z_/f3rpfjgn6j1bwmghgxsm9nbm0000gn/T/ipykernel_39802/1214880203.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(0.0)


In [34]:
final_mongo_df = final_mongo_df.dropna()
final_mongo_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2610 entries, 0 to 2615
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      2610 non-null   object 
 1   product_code    2610 non-null   object 
 2   price_cash      2610 non-null   object 
 3   price_credit    2610 non-null   object 
 4   price_card      2610 non-null   object 
 5   main_boost      2610 non-null   int64  
 6   low_value_flag  2610 non-null   int64  
 7   popularity      2610 non-null   float64
 8   link            2610 non-null   object 
dtypes: float64(1), int64(2), object(6)
memory usage: 203.9+ KB


In [ ]:
# # Select the database and collection
# cm_env = os.environ.get('CM_ENV', 'dev')
# if cm_env == 'prod':
#     collection = client["campomaq"]["cm_catalog"]
# else:
#     collection = client["campomaq_test"]["cm_catalog"]

# # New field to indicate that the product is shown in the catalog
# top_100_codes = (
#     df_top_sales_prods.sort_values("popularity", ascending=False)
#     .head(100)["product_code"]
#     .tolist()
# )

# # Assign True for top 100, False for others
# df_top_sales_prods["show_in_app"] = df_top_sales_prods["product_code"].isin(top_100_codes)

# from pymongo import UpdateOne

# # Build bulk operations
# operations = [
#     UpdateOne(
#         {"product_code": row["product_code"]},
#         {"$set": {"show_in_app": bool(row["show_in_app"])}}
#     )
#     for _, row in df_top_sales_prods.iterrows()
# ]
# # Execute in bulk
# if operations:
#     result = collection.bulk_write(operations, ordered=False)
#     print(f"Matched: {result.matched_count}, Modified: {result.modified_count}")



Matched: 2610, Modified: 162


In [36]:
from pymongo import UpdateOne

operations = [
    UpdateOne(
        {"product_code": record["product_code"]},
        {"$set": record},
        upsert=True  # Key difference
    )
    for record in final_mongo_df.to_dict(orient="records")
]

# Execute in bulk
if operations:
    result = collection.bulk_write(operations, ordered=False)
    print(f"Matched: {result.matched_count}")
    print(f"Modified: {result.modified_count}")
    print(f"Upserted: {result.upserted_count}")

Matched: 2610
Modified: 2610
Upserted: 0
